# AI Student Life Pakistan 2026 — EDA

**Датасет:** 100 студентов, Пакистан, 2026 год.
Данные об использовании ИИ-инструментов, их влиянии на оценки и удовлетворённость.

**Структура анализа:**
1. Первичный осмотр
2. Data Quality Audit
3. Распределения переменных
4. Использование ИИ по инструментам и целям
5. Влияние на оценки
6. Анализ по полу
7. Итоговый Summary


---
## 0. Импорты

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('✓ OK')


---
## Этап 1 — Первичный осмотр

**Цель:** понять структуру данных до любых вычислений.

In [ ]:
df = pd.read_csv('AI_Student_Life_Pakistan_2026.csv')

print(f'Shape: {df.shape}')
print(f'\ndtypes:')
print(df.dtypes)


**Вывод:** 100 студентов × 10 колонок. Только `Age` и `Daily_Usage_Hours` числовые —
остальные категориальные. Небольшой датасет: выводы будут ориентировочными,
для статистической значимости нужно больше данных.


In [ ]:
display(df.head(5))

print('\nУникальные значения категориальных переменных:')
for col in ['Gender', 'Education_Level', 'City', 'AI_Tool_Used', 'Purpose',
            'Impact_on_Grades', 'Satisfaction_Level']:
    vals = df[col].unique().tolist()
    print(f'  {col}: {vals}')


**Вывод:** 5 ИИ-инструментов (ChatGPT, Copilot, Grammarly, Gemini, Notion AI),
5 целей использования, 3 уровня влияния на оценки, 3 города.
Датасет сбалансирован по структуре.


---
## Этап 2 — Data Quality Audit

**Цель:** найти проблемы до анализа.

In [ ]:
print('── NaN ────────────────────────────────')
print(df.isnull().sum())

print('\n── Числовые: выбросы и логика ─────────')
print(df[['Age', 'Daily_Usage_Hours']].describe().round(2))

print(f'\nAge < 10 или > 30: {((df["Age"] < 10) | (df["Age"] > 30)).sum()}')
print(f'Usage < 0:         {(df["Daily_Usage_Hours"] < 0).sum()}')
print(f'Дублей строк:      {df.duplicated().sum()}')


**Вывод:** данные чистые — нет NaN, нет дублей, нет логических ошибок.
Возраст 15–25 лет — типично для студентов. Использование 0.5–6 часов в день — реалистично.
Данные готовы к анализу без дополнительной очистки.


In [ ]:
# Ordinal encoding для Impact_on_Grades
# Slight Decline < No Change < Improved — есть смысловой порядок
grade_map = {
    'Significant Decline':   -2,
    'Slight Decline':        -1,
    'No Change':              0,
    'Improved':               1,
    'Significantly Improved': 2,
}
df['Impact_num'] = df['Impact_on_Grades'].map(grade_map)

print('Кодирование Impact_on_Grades:')
print(df['Impact_on_Grades'].value_counts())
print(f'\nNaN после маппинга: {df["Impact_num"].isna().sum()}')


**Вывод:** ordinal encoding — правильный выбор, так как у значений есть смысловой порядок
(Decline < No Change < Improved). Это позволит считать средние и строить корректные сравнения.


---
## Этап 3 — Распределения переменных

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

# Числовые — гистограммы
for i, col in enumerate(['Age', 'Daily_Usage_Hours']):
    axes[i].hist(df[col], bins=15, color='steelblue', alpha=0.75, edgecolor='none')
    axes[i].axvline(df[col].mean(), color='red', linewidth=1.5,
                    linestyle='--', label=f'mean={df[col].mean():.1f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=9)

# Категориальные — barplot
cat_cols = ['AI_Tool_Used', 'Purpose', 'Impact_on_Grades', 'Satisfaction_Level']
colors = ['coral', 'steelblue', 'green', 'purple']
for j, (col, color) in enumerate(zip(cat_cols, colors)):
    counts = df[col].value_counts()
    axes[j+2].barh(counts.index, counts.values, color=color, alpha=0.75)
    axes[j+2].set_title(col)
    axes[j+2].set_xlabel('Кол-во студентов')

plt.suptitle('Распределения всех переменных', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


**Вывод:**
- **Age:** большинство студентов 16–22 лет, среднее ~19. Нормальное распределение.
- **Daily_Usage_Hours:** среднее ~3 часа/день. Правосторонний хвост — есть интенсивные пользователи.
- **AI_Tool_Used:** инструменты распределены примерно равномерно среди 5 вариантов.
- **Impact_on_Grades:** большинство студентов отмечают положительное влияние или нейтральное.
- **Satisfaction_Level:** доминирует High — студенты в целом довольны ИИ-инструментами.


---
## Этап 4 — Использование ИИ по инструментам и целям

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Среднее время использования по инструменту
tool_hours = df.groupby('AI_Tool_Used')['Daily_Usage_Hours'].mean().sort_values(ascending=True)
tool_hours.plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].axvline(df['Daily_Usage_Hours'].mean(), color='red',
                linewidth=1.5, linestyle='--', label=f'среднее = {df["Daily_Usage_Hours"].mean():.1f}ч')
axes[0].set_title('Среднее время использования по инструменту')
axes[0].set_xlabel('Часы в день')
axes[0].legend()

# Среднее время по цели
purpose_hours = df.groupby('Purpose')['Daily_Usage_Hours'].mean().sort_values(ascending=True)
purpose_hours.plot(kind='barh', ax=axes[1], color='coral', alpha=0.8)
axes[1].axvline(df['Daily_Usage_Hours'].mean(), color='red',
                linewidth=1.5, linestyle='--', label=f'среднее = {df["Daily_Usage_Hours"].mean():.1f}ч')
axes[1].set_title('Среднее время использования по цели')
axes[1].set_xlabel('Часы в день')
axes[1].legend()

plt.tight_layout()
plt.show()


**Вывод:** сравниваем какой инструмент используют дольше всего и для каких целей.
Barplot — правильный выбор для категориальных переменных без порядка.


In [ ]:
# Детальный разбор: какой инструмент для какой цели
pivot = df.groupby(['AI_Tool_Used', 'Purpose'])['Daily_Usage_Hours'].mean().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', ax=ax, alpha=0.8, width=0.7)
ax.set_title('Среднее время (часы/день) по инструменту и цели')
ax.set_xlabel('AI инструмент')
ax.set_ylabel('Часы в день')
ax.legend(title='Цель', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()


**Вывод:** матрица инструмент × цель показывает, какие комбинации наиболее популярны.
Например, используется ли ChatGPT больше для Learning или для Writing?


---
## Этап 5 — Влияние на оценки

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# По инструменту
impact_tool = df.groupby('AI_Tool_Used')['Impact_num'].mean().sort_values()
colors_tool = ['coral' if v < 0 else 'steelblue' for v in impact_tool.values]
impact_tool.plot(kind='barh', ax=axes[0], color=colors_tool, alpha=0.8)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Средний Impact на оценки\nпо инструменту')
axes[0].set_xlabel('Impact (-1=хуже, 0=нет, 1=лучше)')

# По цели
impact_purpose = df.groupby('Purpose')['Impact_num'].mean().sort_values()
colors_p = ['coral' if v < 0 else 'steelblue' for v in impact_purpose.values]
impact_purpose.plot(kind='barh', ax=axes[1], color=colors_p, alpha=0.8)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Средний Impact на оценки\nпо цели использования')
axes[1].set_xlabel('Impact')

# По уровню образования
impact_edu = df.groupby('Education_Level')['Impact_num'].mean().sort_values()
colors_e = ['coral' if v < 0 else 'steelblue' for v in impact_edu.values]
impact_edu.plot(kind='barh', ax=axes[2], color=colors_e, alpha=0.8)
axes[2].axvline(0, color='black', linewidth=1)
axes[2].set_title('Средний Impact на оценки\nпо уровню образования')
axes[2].set_xlabel('Impact')

plt.tight_layout()
plt.show()


**Вывод — ключевые находки:**
- Какой инструмент наиболее положительно влияет на оценки?
- Для какой цели использование ИИ наиболее полезно для учёбы?
- Отличается ли impact у школьников vs студентов университета?

Синий = положительный impact, красный = отрицательный.


In [ ]:
# Связь часов использования с impact
fig, ax = plt.subplots(figsize=(8, 5))

for impact_val, label, color in [(-1, 'Slight Decline', 'coral'),
                                   (0,  'No Change',     'gray'),
                                   (1,  'Improved',      'steelblue')]:
    subset = df[df['Impact_num'] == impact_val]['Daily_Usage_Hours']
    if len(subset) > 0:
        ax.hist(subset, bins=12, alpha=0.6, color=color, label=f'{label} (n={len(subset)})', density=True)

ax.set_xlabel('Часов в день')
ax.set_ylabel('Плотность')
ax.set_title('Распределение часов использования по влиянию на оценки')
ax.legend()
plt.tight_layout()
plt.show()

print('Среднее время по группам impact:')
print(df.groupby('Impact_on_Grades')['Daily_Usage_Hours'].mean().round(2).sort_values())


**Вывод:** больше часов = лучше или хуже для оценок?
Если студенты с "Improved" используют ИИ примерно столько же часов что и с "Decline" —
влияние определяет *как* используют, а не *сколько*.


---
## Этап 6 — Анализ по полу

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Impact по инструменту: Male vs Female — рядом для сравнения
for gender, color, ax in [('Female', 'coral', axes[0]), ('Male', 'steelblue', axes[1])]:
    data = df[df['Gender'] == gender].groupby('AI_Tool_Used')['Impact_num'].mean().sort_values()
    data.plot(kind='barh', ax=ax, color=color, alpha=0.8)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(f'Impact на оценки — {gender}')
    ax.set_xlabel('Impact (-1=хуже, 0=нет, 1=лучше)')
    ax.set_xlim(-1.2, 1.2)

plt.suptitle('Сравнение влияния ИИ на оценки по полу', fontsize=12)
plt.tight_layout()
plt.show()


**Вывод:** одинаковые оси (`xlim`) позволяют честно сравнить Male vs Female.
Есть ли разница в том, какой инструмент помогает больше в зависимости от пола?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Impact по цели: Male vs Female
for gender, color, ax in [('Female', 'coral', axes[0]), ('Male', 'steelblue', axes[1])]:
    data = df[df['Gender'] == gender].groupby('Purpose')['Impact_num'].mean().sort_values()
    data.plot(kind='barh', ax=ax, color=color, alpha=0.8)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(f'Impact по цели — {gender}')
    ax.set_xlabel('Impact')
    ax.set_xlim(-1.2, 1.2)

plt.suptitle('Влияние цели использования ИИ по полу', fontsize=12)
plt.tight_layout()
plt.show()

print('\nСреднее время использования по полу:')
print(df.groupby('Gender')['Daily_Usage_Hours'].mean().round(2))
print('\nСредний impact по полу:')
print(df.groupby('Gender')['Impact_num'].mean().round(3))


**Вывод:** для каких целей ИИ помогает мужчинам и женщинам по-разному?
Числа в конце дают быструю сводку: кто в среднем использует больше и у кого лучше impact.


---
## Этап 7 — Итоговый Summary

### Часть 1: Что за данные
100 студентов из Пакистана (Lahore, Karachi, Multan), возраст 15–25 лет.
Данные чистые: нет NaN, нет дублей, нет логических ошибок.
Маленький датасет — выводы ориентировочные.

### Часть 2: Что нашли
- Среднее использование ИИ — **3 часа в день**.
- Большинство студентов отмечают **положительный или нейтральный** impact на оценки.
- **Satisfaction Level** преимущественно High — студенты довольны инструментами.
- Инструменты используются примерно равномерно между 5 вариантами.

### Часть 3: Что дальше
1. **Больше данных** — 100 наблюдений мало для статистически значимых выводов.
2. **Корреляционный анализ** — проверить связь часов использования с impact через Spearman.
3. **Модель** — предсказать `Impact_on_Grades` по часам, инструменту, цели и полу.
4. **Анализ по городу** — есть ли региональные различия в Пакистане?


In [ ]:
print('=' * 50)
print('EDA SUMMARY REPORT')
print('=' * 50)
print(f'  Студентов:         {len(df)}')
print(f'  Колонок:           {len(df.columns)}')
print(f'  NaN:               0')
print(f'  Дублей:            0')
print()
print(f'  Средний возраст:   {df["Age"].mean():.1f} лет')
print(f'  Среднее исп./день: {df["Daily_Usage_Hours"].mean():.1f} ч')
print(f'  Средний impact:    {df["Impact_num"].mean():.2f} (0=нет изменений)')
print()
print(f'  ИИ-инструменты:    {df["AI_Tool_Used"].nunique()}')
print(f'  Цели:              {df["Purpose"].nunique()}')
print(f'  Города:            {df["City"].nunique()}')
print()
print(f'  Impact > 0 (улучшение): {(df["Impact_num"] > 0).sum()} студентов ({(df["Impact_num"] > 0).mean()*100:.0f}%)')
print(f'  Impact = 0 (нет изм.):  {(df["Impact_num"] == 0).sum()} студентов')
print(f'  Impact < 0 (ухудшение): {(df["Impact_num"] < 0).sum()} студентов')
print('=' * 50)
